# Notebook 15 — Speculative Decoding and Assisted Generation

Speculative decoding only helps when the draft path is cheap enough and the target model accepts enough of the proposed tokens. This notebook studies those conditions directly with notebook-local simulations, plus a second pattern: assisted generation for scaffold-heavy or structured outputs.

## What you will build

- a target-only generation baseline for several workload types
- a speculative-decoding simulator with controllable acceptance rates and coordination overhead
- a proposal-size and draft-model sweep to find useful operating regions
- an assisted-generation model for scaffold-heavy responses
- a runtime controller that disables speculation when live acceptance collapses

## Why runtime behavior matters here

Speculation is not a free speedup. It trades extra draft work, verification logic, and more moving parts for fewer expensive target-model decode steps. The right question is never "is speculation advanced?" The right question is "does it improve the end-to-end latency and cost profile for this workload?"

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Add notebook helpers and an artifact directory

We will keep the math transparent so every speedup claim can be traced back to acceptance rate, proposal size, and coordination overhead.

In [ ]:
random.seed(15)

ARTIFACT_DIR = ARTIFACT_ROOT / "15_speculative_decoding_and_assisted_generation"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def percentile(values, pct):
    values = [float(v) for v in values]
    if not values:
        return 0.0
    return float(np.percentile(values, pct))

def clipped(value, low, high):
    return max(low, min(high, value))

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define workload and model profiles

Different workloads tolerate speculation differently. Predictable JSON extraction behaves very differently from open-ended code editing or long-context analysis.

In [ ]:
workloads = [
    {"workload": "support_chat", "prompt_tokens": 960, "output_tokens": 180, "difficulty": 0.36, "structure_share": 0.22},
    {"workload": "json_extract", "prompt_tokens": 720, "output_tokens": 110, "difficulty": 0.20, "structure_share": 0.84},
    {"workload": "code_edit", "prompt_tokens": 1500, "output_tokens": 210, "difficulty": 0.54, "structure_share": 0.40},
    {"workload": "long_context_summary", "prompt_tokens": 5800, "output_tokens": 170, "difficulty": 0.48, "structure_share": 0.28},
    {"workload": "tool_routing", "prompt_tokens": 840, "output_tokens": 90, "difficulty": 0.26, "structure_share": 0.92},
]

target_profile = {"name": "target_32b", "prefill_tps": 9400, "decode_tps": 46, "step_overhead_ms": 22}
draft_profiles = [
    {"name": "tiny_draft", "draft_tps": 240, "quality_bias": -0.08, "coordination_ms": 8},
    {"name": "balanced_draft", "draft_tps": 190, "quality_bias": 0.03, "coordination_ms": 10},
    {"name": "aggressive_draft", "draft_tps": 155, "quality_bias": 0.08, "coordination_ms": 13},
]

workloads_df = pd.DataFrame(workloads)
draft_profiles_df = pd.DataFrame(draft_profiles)
workloads_df

## Step 3: Compute a target-only baseline

Every advanced decode strategy should be compared with the simplest honest baseline: let the target model do all the work.

In [ ]:
def simulate_target_only(record, target):
    prefill_ms = 1000 * record["prompt_tokens"] / target["prefill_tps"]
    decode_ms = 1000 * record["output_tokens"] / target["decode_tps"]
    total_ms = prefill_ms + decode_ms + target["step_overhead_ms"]
    return {
        "prefill_ms": round(prefill_ms, 2),
        "decode_ms": round(decode_ms, 2),
        "total_ms": round(total_ms, 2),
    }

baseline_rows = []
for record in workloads:
    baseline_rows.append({**record, **simulate_target_only(record, target_profile)})

baseline_df = pd.DataFrame(baseline_rows)
baseline_df[["workload", "prompt_tokens", "output_tokens", "total_ms"]]

## Step 4: Implement a speculative-decoding simulator

The simulator separates three levers: how fast the draft model is, how often it is right, and how many tokens it proposes per round.

In [ ]:
def acceptance_rate(record, draft_profile, proposal_size):
    base = 0.86
    base -= 0.38 * record["difficulty"]
    base += 0.16 * record["structure_share"]
    base += 0.10 * draft_profile["quality_bias"]
    base -= 0.012 * max(proposal_size - 4, 0)
    base -= 0.03 * (record["prompt_tokens"] > 4096)
    return clipped(base, 0.18, 0.95)

def simulate_speculative(record, draft_profile, target, proposal_size=4, network_overhead_ms=0):
    base = simulate_target_only(record, target)
    accept = acceptance_rate(record, draft_profile, proposal_size)
    rounds = max(1, math.ceil(record["output_tokens"] / proposal_size))
    accepted_tokens = record["output_tokens"] * accept
    rejected_tokens = max(record["output_tokens"] - accepted_tokens, 0)
    draft_ms = 1000 * record["output_tokens"] / draft_profile["draft_tps"]
    verify_ms = 1000 * (accepted_tokens * 0.38 + rejected_tokens * 1.12) / target["decode_tps"]
    coordination_ms = rounds * (draft_profile["coordination_ms"] + network_overhead_ms)
    total_ms = base["prefill_ms"] + draft_ms + verify_ms + coordination_ms + target["step_overhead_ms"]
    return {
        "draft_name": draft_profile["name"],
        "proposal_size": proposal_size,
        "acceptance_rate": round(accept, 3),
        "accepted_tokens": round(accepted_tokens, 1),
        "rejected_tokens": round(rejected_tokens, 1),
        "total_ms": round(total_ms, 2),
        "speedup_vs_target": round(base["total_ms"] / max(total_ms, 1e-6), 3),
    }

## Step 5: Sweep proposal sizes

Larger proposal blocks reduce coordination rounds, but they also lower acceptance when the draft model starts guessing too far ahead.

In [ ]:
proposal_rows = []
for record in workloads:
    draft_profile = next(profile for profile in draft_profiles if profile["name"] == "balanced_draft")
    for proposal_size in [2, 4, 6, 8]:
        proposal_rows.append({
            "workload": record["workload"],
            **simulate_speculative(record, draft_profile, target_profile, proposal_size=proposal_size),
        })

proposal_df = pd.DataFrame(proposal_rows)
proposal_df.head(12)

## Step 6: Compare draft model choices

A faster but inaccurate draft can lose to a slightly slower draft with better acceptance. This is the main runtime trade-off behind assisted generation stacks that use tiny local models as helpers.

In [ ]:
draft_rows = []
for record in workloads:
    for profile in draft_profiles:
        draft_rows.append({
            "workload": record["workload"],
            **simulate_speculative(record, profile, target_profile, proposal_size=4),
        })

draft_compare_df = pd.DataFrame(draft_rows)
draft_compare_df.sort_values(["workload", "speedup_vs_target"], ascending=[True, False])

## Step 7: Summarize the best speculative setup per workload

The best setup is not universal. Structured tasks often want a different draft and proposal size than creative or long-context tasks.

In [ ]:
spec_grid_rows = []
for record in workloads:
    for profile in draft_profiles:
        for proposal_size in [2, 4, 6, 8]:
            spec_grid_rows.append({
                "workload": record["workload"],
                **simulate_speculative(record, profile, target_profile, proposal_size=proposal_size),
            })

spec_grid_df = pd.DataFrame(spec_grid_rows)
best_spec_df = spec_grid_df.sort_values("speedup_vs_target", ascending=False).groupby("workload", as_index=False).first()
best_spec_df[["workload", "draft_name", "proposal_size", "acceptance_rate", "speedup_vs_target"]]

## Step 8: Visualize the speculation frontier

The chart below makes the main decision surface visible: speedup only appears when acceptance stays high enough to repay draft and coordination costs.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for draft_name, frame in spec_grid_df.groupby("draft_name"):
    frontier = frame.groupby("proposal_size").agg(speedup_vs_target=("speedup_vs_target", "mean"), acceptance_rate=("acceptance_rate", "mean")).reset_index()
    ax.plot(frontier["acceptance_rate"], frontier["speedup_vs_target"], marker="o", label=draft_name)
ax.set_xlabel("mean acceptance rate")
ax.set_ylabel("mean speedup vs target only")
ax.set_title("Speculation only wins in a bounded acceptance region")
ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
ax.legend()
plt.tight_layout()

## Step 9: Simulate assisted generation

Assisted generation is a sibling pattern. Instead of drafting every token, a helper model or rules engine fills in predictable scaffolds: JSON keys, tool wrappers, or template-heavy language. The target model only spends real decode budget on the less predictable parts.

In [ ]:
def simulate_assisted_generation(record, target, assistant_tps=320, assistant_accuracy=0.91, handoff_ms=18):
    scaffold_tokens = record["output_tokens"] * clipped(record["structure_share"] * 0.82, 0.0, 0.75)
    usable_scaffold_tokens = scaffold_tokens * assistant_accuracy
    target_tokens = max(record["output_tokens"] - usable_scaffold_tokens, 8)
    prefill_ms = 1000 * record["prompt_tokens"] / target["prefill_tps"]
    assistant_ms = 1000 * scaffold_tokens / assistant_tps
    target_ms = 1000 * target_tokens / target["decode_tps"]
    total_ms = prefill_ms + assistant_ms + target_ms + handoff_ms + target["step_overhead_ms"]
    base = simulate_target_only(record, target)
    return {
        "assistant_tokens": round(usable_scaffold_tokens, 1),
        "target_tokens_after_handoff": round(target_tokens, 1),
        "total_ms": round(total_ms, 2),
        "speedup_vs_target": round(base["total_ms"] / max(total_ms, 1e-6), 3),
    }

assisted_rows = []
for record in workloads:
    assisted_rows.append({"workload": record["workload"], **simulate_assisted_generation(record, target_profile)})

assisted_df = pd.DataFrame(assisted_rows)
assisted_df

## Step 10: Compare assisted generation with the best speculative setup

For structured tasks, assistance can be simpler and more stable than full speculation. For open-ended tasks, speculation may still dominate if acceptance stays high.

In [ ]:
compare_modes_df = best_spec_df[["workload", "speedup_vs_target", "acceptance_rate"]].rename(columns={"speedup_vs_target": "best_spec_speedup"})
compare_modes_df = compare_modes_df.merge(assisted_df[["workload", "speedup_vs_target"]].rename(columns={"speedup_vs_target": "assisted_speedup"}), on="workload", how="left")
compare_modes_df["preferred_mode"] = np.where(compare_modes_df["assisted_speedup"] > compare_modes_df["best_spec_speedup"], "assisted_generation", "speculative_decoding")
compare_modes_df

## Step 11: Add coordination overhead sensitivity

Draft and target may not run in the same process or on the same node. A few extra milliseconds per round can erase the win, especially when proposal sizes are small.

In [ ]:
network_rows = []
record = next(item for item in workloads if item["workload"] == "support_chat")
draft_profile = next(profile for profile in draft_profiles if profile["name"] == "balanced_draft")
for overhead_ms in [0, 4, 8, 12, 20]:
    result = simulate_speculative(record, draft_profile, target_profile, proposal_size=4, network_overhead_ms=overhead_ms)
    network_rows.append({"network_overhead_ms": overhead_ms, **result})

network_df = pd.DataFrame(network_rows)
network_df[["network_overhead_ms", "acceptance_rate", "total_ms", "speedup_vs_target"]]

## Step 12: Watch acceptance collapse under domain shift

Live traffic changes. If the request mix shifts from structured extraction to messy novel prompts, the runtime should notice the acceptance collapse and disable speculation instead of paying the overhead forever.

In [ ]:
controller_rows = []
rolling_enabled = True
for window_id in range(18):
    if window_id < 8:
        live_acceptance = 0.78 + random.uniform(-0.04, 0.03)
    elif window_id < 13:
        live_acceptance = 0.63 + random.uniform(-0.05, 0.03)
    else:
        live_acceptance = 0.46 + random.uniform(-0.06, 0.02)
    rolling_enabled = rolling_enabled and live_acceptance >= 0.52
    controller_rows.append({
        "window_id": window_id,
        "live_acceptance": round(live_acceptance, 3),
        "speculation_enabled": rolling_enabled,
        "decision": "disable" if not rolling_enabled else "keep_enabled",
    })

controller_df = pd.DataFrame(controller_rows)
controller_df

## Step 13: Build an operational scorecard

The scorecard below turns the notebook into deployment guidance. We want something a runtime engineer can use to decide whether to enable speculation, use assistance, or stick with plain target decoding.

In [ ]:
scorecard_rows = []
for record in workloads:
    baseline = next(row for row in baseline_rows if row["workload"] == record["workload"])
    best_spec = best_spec_df[best_spec_df["workload"] == record["workload"]].iloc[0]
    assisted = assisted_df[assisted_df["workload"] == record["workload"]].iloc[0]
    recommendation = "target_only"
    if best_spec["speedup_vs_target"] > 1.10 and best_spec["acceptance_rate"] >= 0.60:
        recommendation = "speculative_decoding"
    elif assisted["speedup_vs_target"] > 1.05 and record["structure_share"] >= 0.55:
        recommendation = "assisted_generation"
    scorecard_rows.append({
        "workload": record["workload"],
        "baseline_ms": baseline["total_ms"],
        "best_spec_speedup": round(float(best_spec["speedup_vs_target"]), 3),
        "best_spec_acceptance": round(float(best_spec["acceptance_rate"]), 3),
        "assisted_speedup": round(float(assisted["speedup_vs_target"]), 3),
        "recommended_mode": recommendation,
    })

scorecard_df = pd.DataFrame(scorecard_rows)
scorecard_df

## Step 14: Export artifacts for later routing or release notebooks

Later operational notebooks need a compact representation of which workloads benefit from which decode strategy. We will export the scorecard plus the live controller state.

In [ ]:
artifact = {
    "notebook": "15_speculative_decoding_and_assisted_generation",
    "baseline": baseline_df.to_dict("records"),
    "best_speculation": best_spec_df.to_dict("records"),
    "assisted_generation": assisted_df.to_dict("records"),
    "mode_compare": compare_modes_df.to_dict("records"),
    "network_sensitivity": network_df.to_dict("records"),
    "controller_windows": controller_df.to_dict("records"),
    "scorecard": scorecard_df.to_dict("records"),
}

artifact_path = ARTIFACT_DIR / "decode_strategy_scorecard.json"
write_json(artifact_path, artifact)
print("Wrote:", artifact_path.resolve())

## Recap

Speculative decoding is best understood as a runtime bargain: cheap draft steps are only valuable when the target model accepts enough of them. Assisted generation is a separate bargain: spend helper compute only on the predictable parts of the response. The best production systems often support both and switch between them by workload.

In [ ]:
scorecard_size = len(scorecard_df)
assert scorecard_size == len(workloads)
assert compare_modes_df["preferred_mode"].isin(["assisted_generation", "speculative_decoding"]).all()
assert controller_df["speculation_enabled"].isin([True, False]).all()
print("Notebook 15 sanity checks passed.")